# KNN Imputer
- Based on KNN algorithm
- Each missing vals are imputed using mean val from n_neighbors nearest neighbors found in training set.
- In this we fill missing val of a row with the help of the row which is similar to that row whose val is missing
1. In this we calculate the nearest neigbour distance by using nan_euclidian_distance and replace missing val with the val of nearest neigbour row
2. FOrmula : 
- Distance(x-y) b.w neighbour = sqrt((Total no of pairs/No of pairs present or not missing) * (x2-y2)^2 + (x3-y3)^2)
- dist(x,y) = sqrt(weight * sq distance from present coordinates), where wt = (total no of coord) / (no of present coord)
3. Advantage : More acurate
4. DisAdv : More no of calculation means low speed & need to push trainig data on server for predicting

> Best technique than other imputation technique

In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [2]:
titanic = pd.read_csv(r'C:\Users\HP\OneDrive\Desktop\python\pyCourse\csvFiles\titanic\train.csv', usecols=['Age','Pclass','Fare','Survived'])

In [3]:
titanic.head()

,Survived,Pclass,Age,Fare
0,0,3,22.0,7.2500
1,1,1,38.0,71.2833
2,1,3,26.0,7.9250
3,1,1,35.0,53.1000
4,0,3,35.0,8.0500


In [4]:
titanic.isnull().mean()*100

Survived     0.00000
Pclass       0.00000
Age         19.86532
Fare         0.00000
dtype: float64

In [7]:
a = titanic.drop(columns=['Survived'])
b = titanic['Survived']

In [10]:
a_tr, a_ts, b_tr, b_ts = train_test_split(a,b,test_size=0.2, random_state=2)

- Weights have two vals: uniform & distance
- Uniform means simply calculating mean of nearest neighbr val and filling inplace of missing
- But in Distance, we mutliply 1/distance factor to individual val while calculating mean
- Due to this nearest ngbr get more weightage than far val while calculating mean

In [24]:
# Here n_neighbors default val is 5, so we can change it for better res
# Here weights default val is uniform, we can do distance also
knn = KNNImputer(n_neighbors=1, weights='distance')
a_tr_trf = knn.fit_transform(a_tr)
a_ts_trf = knn.transform(a_ts)

In [14]:
# pd.DataFrame(a_tr_trf, columns=a_tr.columns)

In [25]:
lr = LogisticRegression()
lr.fit(a_tr_trf, b_tr)
b_pred = lr.predict(a_ts_trf)
accuracy_score(b_ts, b_pred)

0.7206703910614525

Conparison with Simple Mean Imputer

In [26]:
si = SimpleImputer()
a_tr_trf2 = si.fit_transform(a_tr)
a_ts_trf2 = si.transform(a_ts)

In [27]:
lr = LogisticRegression()
lr.fit(a_tr_trf2, b_tr)
a_pred = lr.predict(a_ts_trf2)
accuracy_score(b_ts, a_pred)

0.6927374301675978

> <h4>Values can be Missing in Three Ways:</h4>
1. MCAR - Missing Completely At Random means data collectn not occured for certain cols
2. MAR - Missing At Random means some values are left missing optionally but missing vals can be filled using other cols
3. MNAR - Missing Not At Random means intentionally values are left missing or removed from dataset and vals are left in such a way that you can't predict them

# Iterative Imputer / MICE
- MICE - Multivariate Imputatn by Chained Equations
- Assumptions : Data Should be MAR means Missing Values are missing At Random and they can be filled by using other cols of dataset
- Adv : Accurate
- DisAdvantage : Slow as we are predicting missing vals using model and memeory issue as we need to store training data on server
- It is applied Only on Input cols

> How to do it?
1. FIll all the NAN vals with mean of respective cols
2. Now move left to right and remove that val of col1 which is filled in step1 to predict that val
3. Predict the missing val of col1 using other cols or neightbr cols which will behave as training data 
4. Now remove all col2 missing vals(next cols)
5. Predict the missing val of col2 using other cols by training model using other cols vals as training data
6. Now repeat for all remainng cols
7. Now do this whole iteration one more time 
8. And subtract iteratn 2 res from iteratn 1 and see the difference
9. Now repeat this whole process of doing 2 iteratn by taking iteration 2 data as base for next iteratn 1 and then checking results till difference val of missing val become zero